# Fine-tuning Supervisionado (SFT + LoRA/QLoRA)

Esqueleto do notebook para treinar o adaptador sobre **Qwen/Qwen2.5-Coder-1.5B-Instruct** no Google Colab (ex.: GPU T4), **após** a coleta com o professor (`Qwen/Qwen2.5-14B-Instruct-AWQ`), filtragem (`F1–F5`), medição de baselines e congelamento (`passo1-freeze`).

## Fundamentação Matemática (LoRA e QLoRA)

- **Adaptação de baixo posto (LoRA):**
  $$W' = W + \Delta W, \qquad \Delta W = \frac{\alpha}{r}BA, \qquad A \in \mathbb{R}^{r\times k}, \ B \in \mathbb{R}^{d\times r}, \ r \ll \min(d, k)$$
  onde $W \in \mathbb{R}^{d\times k}$ permanece congelado e $B$ é inicializado em zero (de modo que $\Delta W = 0$ no início do treino).
- **Redução de parâmetros treináveis por matriz:**
  $$\frac{r(d+k)}{dk}$$
  Por exemplo, para $d = k = 2048$ e $r = 16$, temos $\frac{16(2048+2048)}{2048 \times 2048} = \frac{65.536}{4.194.304} \approx 1{,}56\%$.
- **QLoRA (4-bit NF4):** os pesos base ficam quantizados em 4-bit (NF4) e congelados, enquanto os parâmetros do adaptador LoRA são treinados em maior precisão; o ganho principal é na memória de GPU, e não na contagem de parâmetros treináveis do adaptador.

## 1. Dependências e Verificação de Ambiente (GPU / Google Drive)

In [ ]:
# Execute no Google Colab se ainda não tiver os pacotes instalados:
# %pip install -q -U torch transformers accelerate peft trl bitsandbytes datasets

import os
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM total: {vram_gb:.2f} GB")

for pkg in ["transformers", "peft", "trl", "bitsandbytes"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"{pkg}: NÃO INSTALADO")

# Opcional no Colab: montar Google Drive para salvar checkpoints a cada época
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Configuração de Hiperparâmetros e Caminhos

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainConfig:
    model_id: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    train_file: str = "datasets/sft_jsonl/train.jsonl"
    val_file: str = "datasets/sft_jsonl/val.jsonl"
    test_file: str = "datasets/sft_jsonl/test.jsonl"
    # Salve em /content/drive/MyDrive/... no Colab para sobreviver a quedas de sessão
    output_dir: str = "checkpoints/qwen25_coder_1_5b_lora"
    seed: int = 42
    max_seq_length: int = 2048
    # LoRA / QLoRA
    use_qlora_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple = ("q_proj", "k_proj", "v_proj", "o_proj")
    # Treino
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8

cfg = TrainConfig()
print(cfg)

## 3. Carregamento dos Splits JSONL (`train`, `val`, `test`)

In [ ]:
import json
from pathlib import Path

def load_jsonl_records(path_str: str):
    p = Path(path_str)
    if not p.exists():
        print(f"[AVISO] Arquivo não encontrado ainda: {p}")
        return []
    rows = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl_records(cfg.train_file)
val_rows = load_jsonl_records(cfg.val_file)
test_rows = load_jsonl_records(cfg.test_file)

print(f"Exemplos carregados -> train: {len(train_rows)}, val: {len(val_rows)}, test: {len(test_rows)}")

## 4. Tokenização, Template de Chat e Máscara de *Completion Only*

Para o modelo aprender a gerar a ação alvo sem receber *loss* sobre o prompt (`system` + `user`), mascaramos os tokens anteriores à resposta do `assistant` com `-100` nos `labels`.

In [ ]:
from transformers import AutoTokenizer

def build_completion_only_example(row: dict, tokenizer, max_length: int = 2048):
    """Formata messages com chat template e mascara prompt (system+user) com -100."""
    messages = row.get("messages", [])
    if len(messages) < 3:
        raise ValueError("Esperado ao menos 3 mensagens: system, user e assistant.")

    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=max_length, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, truncation=True, max_length=max_length, add_special_tokens=False)["input_ids"]

    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    attention_mask = [1] * len(full_ids)

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# Exemplo de uso após carregar o tokenizer:
# tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

## 5. Carregamento do Modelo Base e Configuração LoRA / QLoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_lora_model_and_tokenizer(cfg: TrainConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"device_map": "auto"}
    if cfg.use_qlora_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(cfg.model_id, **model_kwargs)
    model.config.use_cache = False

    if cfg.use_qlora_4bit:
        model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tokenizer

## 6. Treinamento (Checkpoints por Época, Validação e Curvas de Loss)

In [ ]:
# TODO (após passo1-freeze e dados reais em datasets/sft_jsonl/):
# 1. Construir datasets tokenizados de train e val com build_completion_only_example.
# 2. Configurar SFTTrainer / Trainer com save_strategy='epoch', eval_strategy='epoch',
#    load_best_model_at_end=True e metric_for_best_model='eval_loss'.
# 3. Registrar histórico de loss de treino e validação por época para plotar e
#    inspecionar overfitting (ex.: com matplotlib).
# 4. Salvar o adaptador final (não commitar pesos no Git; publicar no Drive/HF e
#    registrar o link no README).

## 7. Inferência e Avaliação no Conjunto de Teste (Nível 1)

In [ ]:
# TODO (após o treino):
# 1. Rodar inferência no split test com temperatura baixa / determinística (ex.: temperature=0.0,
#    do_sample=False) usando o mesmo prompt e parser.
# 2. Calcular Acc_tool, Acc_args, SchemaValid e IC 95% (conforme Passo 5 e Passo 7),
#    além de matriz de confusão e exemplos de falha.